In [1]:
# Environment Setup
!git clone https://github.com/anikanb-32/musicandmemory.git
%cd musicandmemory
!cd /content/musicandmemory && git pull
!pip install -r requirements.txt

fatal: destination path 'musicandmemory' already exists and is not an empty directory.
/content/musicandmemory
Already up to date.


In [2]:
# Connect to Google Drive and OpenAI API
from google.colab import userdata, drive
import os, json, time
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/GenAI Final Project Music&Memory/'

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
client = OpenAI()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Test to see if imported files are empty
!cat /content/musicandmemory/src/retrieval.py
!cat /content/musicandmemory/src/generation.py
!cat /content/musicandmemory/configs/prompts.py

import faiss
import numpy as np
import pandas as pd
from openai import OpenAI

client = OpenAI()

def load_retrieval_system(index_path, kb_path):
    """Load the FAISS index and knowledge base."""
    index = faiss.read_index(index_path)
    df = pd.read_csv(kb_path)
    return index, df

def retrieve(query, index, df, k=20):
    """Retrieve top-k songs for a query string."""
    # Embed the query
    response = client.embeddings.create(
        input=[query], model="text-embedding-3-small"
    )
    query_vec = np.array([response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(query_vec)

    # Search
    scores, indices = index.search(query_vec, k)

    # Return results
    results = df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    return results
from openai import OpenAI
import json

client = OpenAI()

def generate_playlist(profile, retrieved_songs_df, prompt_template):
    """Generate a playlist and caregiver cards from profile + retrieved song

In [4]:
# Test to see if imported files are imported
# from src.profiling import generate_queries, profile_to_context
from src.retrieval import load_retrieval_system, retrieve
from src.generation import generate_playlist
from configs.prompts import GENERATION_PROMPT

print('All imports succeed!')

All imports succeed!


In [8]:
# Replace with Anika's file
# from src.profiling import generate_queries, profile_to_context

QUERY_GENERATION_PROMPT = """You are a music therapy specialist. Given a patient profile, generate
retrieval queries to find personally meaningful songs from their life.

RULES:
1. Focus on the REMINISCENCE BUMP (ages 15–25) — this is when music memories are strongest
2. Consider their geographic region (local radio stations played regional hits)
3. Map life events to time periods (wedding songs, graduation year hits, etc.)
4. Consider cultural background (what genres/artists were popular in their community)
5. Generate 5–8 specific queries

PATIENT PROFILE:
{profile}

Respond with a JSON array of query strings only. Example:
["popular Motown hits in Detroit 1963-1968", "top R&B songs 1965", ...]
"""


def generate_queries(profile):
    """Generate retrieval queries from a patient profile."""
    bump_start = profile['birth_year'] + 15
    bump_end = profile['birth_year'] + 25
    profile_with_bump = {**profile, 'reminiscence_bump': f'{bump_start}-{bump_end}'}

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': 'You generate music retrieval queries. Respond with JSON only.'},
            {'role': 'user', 'content': QUERY_GENERATION_PROMPT.format(
                profile=json.dumps(profile_with_bump, indent=2)
            )},
        ],
        temperature=0.3,
    )

    text = response.choices[0].message.content
    text = text.strip().strip('```json').strip('```').strip()
    queries = json.loads(text)
    return queries


def profile_to_context(profile, index, df, k_per_query=10, total_k=20):
    """Full pipeline: profile → queries → retrieved songs."""
    queries = generate_queries(profile)
    print(f'  Generated {len(queries)} queries')

    all_results = []
    for query in queries:
        results = retrieve(query, index, df, k=k_per_query)
        results['source_query'] = query
        all_results.append(results)

    combined = pd.concat(all_results)
    combined = (
        combined
        .sort_values('similarity_score', ascending=False)
        .drop_duplicates(subset=['song', 'artist'], keep='first')
        .head(total_k)
    )

    return combined, queries

In [9]:
# Load the retrieval system
index, df = load_retrieval_system(DATA_PATH + 'songs.index', DATA_PATH + 'knowledge_base.csv')

# Define a patient
patient = {
    "name": "James Wilson",
    "birth_year": 1948,
    "hometown": "Detroit, Michigan",
    "cultural_background": "African American",
    "life_events": [
        {"year": 1966, "event": "Graduated from Cass Tech High School"},
        {"year": 1968, "event": "Drafted into the Vietnam War"},
        {"year": 1971, "event": "Married Dorothy in Detroit"},
        {"year": 1975, "event": "First child born"},
    ]
}

# Retrieved songs based on profile
retrieved, queries = profile_to_context(patient, index, df)
print(f"Retrieved {len(retrieved)} unique songs")

# Generate playlist
result = generate_playlist(patient, retrieved, GENERATION_PROMPT)

# Display
print("\n=== PLAYLIST ===")
for song in result["playlist"]:
    print(f"  {song['rank']}. {song['song']} — {song['artist']} ({song['year']})")
    print(f"     Reason: {song['relevance']}\n")

print("\n=== CAREGIVER CARDS ===")
for card in result["caregiver_cards"]:
    print(f"  Song: {card['song']}")
    print(f"  Prompt: {card['prompt']}\n")


  Generated 8 queries
Retrieved 20 unique songs

=== PLAYLIST ===
  1. Soul Man — Sam & Dave (1967)
     Reason: This song is a classic soul hit from the patient's reminiscence bump period and aligns with his cultural background as an African American from Detroit, a city known for its rich musical heritage.

  2. Cry Like A Baby — The Box Tops (1968)
     Reason: Released during the year James was drafted into the Vietnam War, this song may evoke memories of that significant time in his life.

  3. Groovin' — The Young Rascals (1967)
     Reason: A popular song from the late 1960s that could remind James of his late teenage years and the cultural vibe of the era.

  4. Little Bit O' Soul — The Music Explosion (1967)
     Reason: This upbeat song from the late 1960s might resonate with James's youthful energy and the vibrant Detroit music scene.

  5. Rock Your Baby — George McCrae (1974)
     Reason: Released around the time of his first child's birth, this song could bring back memor